# **Unit 3 Assignment: Building a Production Advanced RAG System**

## **Install Dependencies**

In [ ]:
!pip install rank-bm25 sentence-transformers groq google-generativeai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 2.1 MB/s eta 0:00:00


## **Imports and Setup**

In [ ]:
import os
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import groq
import google.generativeai as genai
from google.colab import userdata  # Colab secrets
from dotenv import load_dotenv
import getpass

# Gemini
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API Key: ")

# Groq
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

print("Keys loaded.")

import google.generativeai as genai
from groq import Groq

# Gemini
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

# Groq
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Clients initialized.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Enter your Gemini API Key: ··········
Enter your Groq API Key: ··········
Keys loaded.
Clients initialized.


## **Part 1 - Document Corpus**

In [ ]:
corpus = [
    # Transformers & Attention
    "The transformer architecture uses self-attention to weigh the importance of each token relative to others in a sequence.",
    "Attention mechanisms allow models to focus on relevant parts of the input by computing query, key, and value projections.",
    "Multi-head attention runs several attention functions in parallel, letting the model capture different types of relationships simultaneously.",

    # Neural Network Training
    "Backpropagation computes gradients of the loss with respect to each parameter by applying the chain rule of calculus.",
    "Gradient descent updates model weights iteratively in the direction that reduces the loss function.",
    "Batch normalization stabilizes training by normalizing layer inputs, which allows higher learning rates and faster convergence.",

    # Optimization
    "Adam optimizer combines momentum and adaptive learning rates, making it widely used for training deep neural networks.",
    "Learning rate scheduling reduces the learning rate during training, often improving final model performance.",
    "Dropout is a regularization technique that randomly sets neuron activations to zero during training to prevent overfitting.",

    # Embeddings & Representation
    "Word2Vec learns dense vector representations of words by predicting surrounding context words in a sliding window.",
    "BERT uses a masked language modeling objective to pretrain bidirectional representations from unlabeled text.",

    # Technical jargon doc — BM25 should find this well
    "The Scaled Dot-Product Attention formula is: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V, where d_k is the key dimension.",
]

print(f"Corpus size: {len(corpus)} documents")
for i, doc in enumerate(corpus):
    print(f"[{i}] {doc[:80]}...")

Corpus size: 12 documents
[0] The transformer architecture uses self-attention to weigh the importance of each...
[1] Attention mechanisms allow models to focus on relevant parts of the input by com...
[2] Multi-head attention runs several attention functions in parallel, letting the m...
[3] Backpropagation computes gradients of the loss with respect to each parameter by...
[4] Gradient descent updates model weights iteratively in the direction that reduces...
[5] Batch normalization stabilizes training by normalizing layer inputs, which allow...
[6] Adam optimizer combines momentum and adaptive learning rates, making it widely u...
[7] Learning rate scheduling reduces the learning rate during training, often improv...
[8] Dropout is a regularization technique that randomly sets neuron activations to z...
[9] Word2Vec learns dense vector representations of words by predicting surrounding ...
[10] BERT uses a masked language modeling objective to pretrain bidirectional represe...
[11] 

## **Part 2 - Implement Hybrid Retrieval**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k  # RRF constant

        # BM25 index — tokenize on whitespace, lowercase
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        # SBERT index — encode all docs once
        print("Loading SBERT model and encoding corpus...")
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)
        print("HybridRetriever ready.")

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # --- BM25 ranking ---
        bm25_scores = self.bm25.get_scores(query.lower().split())
        # argsort descending → ranks (rank 0 = best)
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_rank = {doc_id: rank for rank, doc_id in enumerate(bm25_ranked)}

        # --- SBERT ranking ---
        query_emb = self.sbert.encode([query], convert_to_numpy=True)
        # cosine similarity

        sbert_scores = cosine_similarity(self.doc_embeddings, query_emb).flatten()
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_rank = {doc_id: rank for rank, doc_id in enumerate(sbert_ranked)}

        # --- RRF fusion ---
        all_doc_ids = range(len(self.corpus))
        rrf_scores = {}
        for doc_id in all_doc_ids:
            r_bm25 = bm25_rank.get(doc_id, len(self.corpus))
            r_sbert = sbert_rank.get(doc_id, len(self.corpus))
            rrf_scores[doc_id] = (
                1 / (self.k + r_bm25) +
                1 / (self.k + r_sbert)
            )

        # Sort by RRF score descending, take top_k
        top_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:top_k]

        return [
            {
                "doc_id": doc_id,
                "rrf_score": rrf_scores[doc_id],
                "bm25_rank": bm25_rank[doc_id],
                "sbert_rank": sbert_rank[doc_id],
                "text": self.corpus[doc_id],
            }
            for doc_id in top_ids
        ]

retriever = HybridRetriever(corpus)

Loading SBERT model and encoding corpus...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HybridRetriever ready.


## **Part 3 — Implement a Cross-Encoder Re-Ranker**

In [ ]:
print("Loading cross-encoder...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder ready.")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-ranks candidates using cross-encoder scores.
    Takes the ORIGINAL user query (not HyDE-expanded).
    """
    pairs = [[query, doc["text"]] for doc in candidates]
    scores = cross_encoder.predict(pairs)

    for doc, score in zip(candidates, scores):
        doc["cross_encoder_score"] = float(score)

    reranked = sorted(candidates, key=lambda x: x["cross_encoder_score"], reverse=True)
    return reranked[:top_k]

Loading cross-encoder...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder ready.


## **Part 4 — Implement HyDE Query Expansion**

In [ ]:
def hyde_expand(user_query: str) -> str:
    """
    Uses Gemini to generate a hypothetical answer document.
    This bridges the vocabulary gap between vague queries and technical docs.
    """
    prompt = (
        f"You are an expert in AI and machine learning.\n"
        f"A student asked: '{user_query}'\n\n"
        f"Write a short, technical explanation (2-4 sentences) that directly answers this question.\n"
        f"Include relevant terminology, definitions, and key concepts.\n"
        f"Do not include conversational phrases."
    )

    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(temperature=0.0)
    )
    hypothetical_doc = response.text.strip()
    return hypothetical_doc

## **Part 5 — End-to-End Pipeline**

In [ ]:
def advanced_rag(user_query: str) -> str:
    """
    Full pipeline: HyDE Query Expansion → Hybrid Retrieval → Re-Ranking → LLM Generation
    Returns the final answer string.
    """
    print(f"\n{'='*60}")
    print(f"USER QUERY: {user_query}")

    # Step 1 — HyDE expansion
    print("\n[Step 1] Generating hypothetical document (HyDE)...")
    hyde_doc = hyde_expand(user_query)
    print(f"HyDE doc: {hyde_doc[:150]}...")

    # Step 2 — Hybrid retrieval using HyDE doc as query
    print("\n[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...")
    candidates = retriever.retrieve(hyde_doc, top_k=5)
    for c in candidates:
        print(f"  doc_id={c['doc_id']} rrf={c['rrf_score']:.5f} bm25_rank={c['bm25_rank']} sbert_rank={c['sbert_rank']}")
        print(f"    (BM25 strong? {c['bm25_rank'] < c['sbert_rank']})")
        print(f"    {c['text'][:80]}...")

    # Step 3 — Re-rank using ORIGINAL query (not HyDE)
    print("\n[Step 3] Cross-encoder re-ranking...")
    top_docs = rerank(user_query, candidates, top_k=3)
    for d in top_docs:
        print(f"  cross_score={d['cross_encoder_score']:.4f} | {d['text'][:80]}...")

    # Step 4 — Generate final answer with Groq
    print("\n[Step 4] Generating answer with Groq...")
    context = "\n".join([f"- {d['text']}" for d in top_docs])
    prompt = (
        f"You are a university AI/ML assistant. Answer the student's question using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Student question: {user_query}\n\n"
        f"Answer clearly and concisely in 2-4 sentences."
    )

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    answer = response.choices[0].message.content.strip()
    print(f"\nFINAL ANSWER:\n{answer}")
    return answer

## **Part 6 — Comparison Experiment**

In [ ]:
def naive_rag(user_query: str) -> str:
    """
    Naïve RAG: Dense-only retrieval (SBERT cosine), no expansion, no re-ranking.
    """
    query_emb = retriever.sbert.encode([user_query], convert_to_numpy=True)
    from sklearn.metrics.pairwise import cosine_similarity
    sbert_scores = cosine_similarity(self.doc_embeddings, query_emb).flatten()
    top_ids = np.argsort(sbert_scores)[::-1][:3]

    top_docs = [corpus[i] for i in top_ids]
    context = "\n".join([f"- {d}" for d in top_docs])

    prompt = (
        f"You are a university AI/ML assistant. Answer the student's question using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Student question: {user_query}\n\n"
        f"Answer clearly and concisely in 2-4 sentences."
    )

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

In [ ]:
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is the formula for scaled dot-product attention?"
]

results = []

for query in test_queries:
    print(f"\n{'#'*60}")
    print(f"QUERY: {query}")

    # Naïve RAG
    naive_emb = retriever.sbert.encode([query], convert_to_numpy=True)
    naive_scores = (retriever.doc_embeddings @ naive_emb.T).flatten()
    naive_top_id = int(np.argsort(naive_scores)[::-1][0])
    naive_top_doc = corpus[naive_top_id]

    # Advanced RAG — just get top doc
    hyde_doc = hyde_expand(query)
    candidates = retriever.retrieve(hyde_doc, top_k=5)
    top_docs = rerank(query, candidates, top_k=3)
    advanced_top_doc = top_docs[0]["text"]

    different = "✅ Yes" if naive_top_doc != advanced_top_doc else "❌ No"

    results.append({
        "query": query,
        "naive_top": naive_top_doc,
        "advanced_top": advanced_top_doc,
        "different": different,
    })

    print(f"\nNaïve top:    {naive_top_doc[:100]}...")
    print(f"Advanced top: {advanced_top_doc[:100]}...")
    print(f"Different?    {different}")


############################################################
QUERY: how do transformers encode meaning?

Naïve top:    The transformer architecture uses self-attention to weigh the importance of each token relative to o...
Advanced top: The transformer architecture uses self-attention to weigh the importance of each token relative to o...
Different?    ❌ No

############################################################
QUERY: optimization techniques for training

Naïve top:    Gradient descent updates model weights iteratively in the direction that reduces the loss function....
Advanced top: Adam optimizer combines momentum and adaptive learning rates, making it widely used for training dee...
Different?    ✅ Yes

############################################################
QUERY: what is the formula for scaled dot-product attention?

Naïve top:    The Scaled Dot-Product Attention formula is: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V, whe...
Advanced top: The Scaled Dot-Produc

## **Part 6 — Comparison Table**

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|---|---|---|---|
| "how do transformers encode meaning?" | The transformer architecture uses self-attention to weigh the importance of each token relative to others in a sequence. | The transformer architecture uses self-attention to weigh the importance of each token relative to others in a sequence. | ❌ No |
| "optimization techniques for training" | Gradient descent updates model weights iteratively in the direction that reduces the loss function. | Adam optimizer combines momentum and adaptive learning rates, making it widely used for training deep neural networks. | ✅ Yes |
| "what is the formula for scaled dot-product attention?" | The Scaled Dot-Product Attention formula is: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V, where d_k is the key dimension. | The Scaled Dot-Product Attention formula is: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V, where d_k is the key dimension. | ❌ No |

### Observations

- **Query 1 ("how do transformers encode meaning?"):** Both pipelines returned the same top document. This is expected — the query is semantically clear enough that SBERT alone retrieves the right document without needing HyDE expansion. The self-attention document is the obvious best match regardless of pipeline.

- **Query 2 ("optimization techniques for training"):** The pipelines diverged here. Naïve RAG returned the *gradient descent* document (a direct keyword match on "training"), while Advanced RAG returned the *Adam optimizer* document — a broader, more complete answer to a question about optimization techniques. HyDE likely generated a hypothetical answer mentioning Adam, momentum, and adaptive learning rates, which shifted retrieval toward the more informative document.

- **Query 3 ("what is the formula for scaled dot-product attention?"):** Both pipelines correctly identified the jargon-heavy formula document. This shows that even Naïve SBERT handles highly specific technical queries well when the document contains exact matching terminology. BM25's strength here was redundant since SBERT already found the right doc.

### Summary

Advanced RAG shows the most benefit on **ambiguous or broad queries** (like Query 2), where HyDE bridges the vocabulary gap by expanding the query into richer technical language before retrieval. For queries that are already precise or have an obvious single best match, both pipelines converge on the same result.

## **Full Pipeline Demo**

In [ ]:
# Full pipeline demo
answer = advanced_rag("how do transformers encode meaning?")


USER QUERY: how do transformers encode meaning?

[Step 1] Generating hypothetical document (HyDE)...
HyDE doc: Transformers encode meaning by first converting input tokens into dense vector embeddings, augmented with positional encodings to capture sequence ord...

[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...
  doc_id=0 rrf=0.03280 bm25_rank=2 sbert_rank=0
    (BM25 strong? False)
    The transformer architecture uses self-attention to weigh the importance of each...
  doc_id=2 rrf=0.03279 bm25_rank=1 sbert_rank=1
    (BM25 strong? False)
    Multi-head attention runs several attention functions in parallel, letting the m...
  doc_id=1 rrf=0.03254 bm25_rank=0 sbert_rank=3
    (BM25 strong? True)
    Attention mechanisms allow models to focus on relevant parts of the input by com...
  doc_id=9 rrf=0.03125 bm25_rank=4 sbert_rank=4
    (BM25 strong? False)
    Word2Vec learns dense vector representations of words by predicting surrounding ...
  doc_id=10 rrf=0.03105 bm25_rank=7 sber

## **Full Pipeline Demo Output Analysis**

**Query:** "how do transformers encode meaning?"

### Step-by-Step Observations

**Step 1 — HyDE Expansion:**
Gemini generated a technically rich hypothetical answer mentioning "dense vector embeddings", "positional encodings", and "sequence order" — vocabulary that does not appear in the original query but closely matches the language used in the corpus. This is exactly the vocabulary gap HyDE is designed to bridge.

**Step 2 — Hybrid Retrieval (BM25 + SBERT + RRF):**
- `doc_id=1` (Attention mechanisms...) was ranked **BM25 rank 0** but only **SBERT rank 3** — meaning BM25 found it via keyword overlap ("attention", "focus") while SBERT considered it less semantically central. RRF fused both signals and still included it in the top 5.
- `doc_id=0` (transformer self-attention) and `doc_id=2` (multi-head attention) were top-ranked by both retrievers, giving them the highest RRF scores (0.03280 and 0.03279 respectively) — very close, as expected with RRF's compression effect.
- `doc_id=10` (BERT) appeared despite a low BM25 rank (7) purely due to SBERT rank 2 — a semantic match that BM25 alone would have missed.

**Step 3 — Cross-Encoder Re-Ranking:**
All three cross-encoder scores are negative (-8.53, -11.22, -11.31), which is normal for this model. The cross-encoder demoted BERT and Word2Vec — both are loosely related but don't directly answer the query — and promoted `doc_id=0` (self-attention) to the top. This shows the re-ranker correctly distinguishing *related* documents from *relevant* ones.

**Step 4 — Final Answer:**
Groq's answer accurately synthesizes the top document, correctly explaining self-attention and contextual dependencies. The answer is coherent, on-topic, and does not hallucinate beyond what the context provides.

## **Bonus Challenge 1 — Weighted RRF (α experiment)**

In [ ]:
def weighted_rrf_retrieve(query: str, alpha: float, top_k: int = 5, k: int = 60) -> list[dict]:
    """
    Weighted RRF: alpha controls BM25 weight, (1-alpha) controls SBERT weight.
    alpha=1.0 → pure BM25, alpha=0.0 → pure SBERT
    """
    # BM25
    bm25_scores = retriever.bm25.get_scores(query.lower().split())
    bm25_ranked = np.argsort(bm25_scores)[::-1]
    bm25_rank = {doc_id: rank for rank, doc_id in enumerate(bm25_ranked)}

    # SBERT
    query_emb = retriever.sbert.encode([query], convert_to_numpy=True)
    from sklearn.metrics.pairwise import cosine_similarity
    sbert_scores = cosine_similarity(retriever.doc_embeddings, query_emb).flatten()
    sbert_ranked = np.argsort(sbert_scores)[::-1]
    sbert_rank = {doc_id: rank for rank, doc_id in enumerate(sbert_ranked)}

    # Weighted RRF
    rrf_scores = {}
    for doc_id in range(len(corpus)):
        r_bm25 = bm25_rank.get(doc_id, len(corpus))
        r_sbert = sbert_rank.get(doc_id, len(corpus))
        rrf_scores[doc_id] = (
            alpha * (1 / (k + r_bm25)) +
            (1 - alpha) * (1 / (k + r_sbert))
        )

    top_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:top_k]
    return [{"doc_id": i, "rrf_score": rrf_scores[i], "text": corpus[i]} for i in top_ids]


# Test on a keyword-heavy query vs a semantic query
keyword_query = "what is the formula for scaled dot-product attention?"
semantic_query = "how do transformers encode meaning?"

print("=" * 60)
print("Bonus 1 — Weighted RRF (α experiment)\n")

for alpha in [0.3, 0.5, 0.7]:
    print(f"\n--- α = {alpha} (BM25 weight={alpha}, SBERT weight={1-alpha}) ---")

    print(f"  [Keyword query] '{keyword_query[:50]}...'")
    results = weighted_rrf_retrieve(keyword_query, alpha=alpha, top_k=1)
    print(f"    Top doc: {results[0]['text'][:100]}...")

    print(f"  [Semantic query] '{semantic_query[:50]}...'")
    results = weighted_rrf_retrieve(semantic_query, alpha=alpha, top_k=1)
    print(f"    Top doc: {results[0]['text'][:100]}...")

Bonus 1 — Weighted RRF (α experiment)


--- α = 0.3 (BM25 weight=0.3, SBERT weight=0.7) ---
  [Keyword query] 'what is the formula for scaled dot-product attenti...'
    Top doc: The Scaled Dot-Product Attention formula is: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V, whe...
  [Semantic query] 'how do transformers encode meaning?...'
    Top doc: BERT uses a masked language modeling objective to pretrain bidirectional representations from unlabe...

--- α = 0.5 (BM25 weight=0.5, SBERT weight=0.5) ---
  [Keyword query] 'what is the formula for scaled dot-product attenti...'
    Top doc: The Scaled Dot-Product Attention formula is: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V, whe...
  [Semantic query] 'how do transformers encode meaning?...'
    Top doc: BERT uses a masked language modeling objective to pretrain bidirectional representations from unlabe...

--- α = 0.7 (BM25 weight=0.7, SBERT weight=0.30000000000000004) ---
  [Keyword query] 'what is the formula for scaled 

## Bonus 1 — Weighted RRF (α experiment) Observations

**Query 1 (Keyword-heavy): "what is the formula for scaled dot-product attention?"**
The formula document was retrieved as the top result across **all three α values** (0.3, 0.5, 0.7). This indicates that both BM25 and SBERT independently agree on this document — BM25 due to exact term matches ("Scaled Dot-Product Attention", "softmax", "sqrt"), and SBERT due to semantic similarity. When both retrievers agree, changing α has no effect on the top result.

**Query 2 (Semantic): "how do transformers encode meaning?"**
Interestingly, BERT (`doc_id=10`) was returned as the top document across all α values, including α=0.3 where SBERT has higher weight. This is a subtle retrieval error — BERT the document is *about* a model named BERT, not directly about how transformers encode meaning. The transformer self-attention document (`doc_id=0`) would have been a more precise answer. This suggests that without HyDE expansion (this experiment queries directly without expansion), SBERT still struggles slightly with the vague phrasing, and BM25 doesn't help since the word "transformers" appears in multiple docs.

**Key Takeaway:**
Changing α had **no observable effect** on the top result for either query in this corpus. This is likely because the corpus is small (12 documents) — with few candidates, BM25 and SBERT tend to agree on the top document, making the weighting redundant. In a larger corpus with more competing documents, α tuning would matter more: higher α (BM25-heavy) would benefit exact keyword queries, and lower α (SBERT-heavy) would benefit vague semantic ones. This experiment also highlights why HyDE is valuable — the semantic query performed better in the full pipeline (Cell 11) where HyDE expansion was used, compared to here where the raw query was passed directly.

## **Bonus Challenge 2 — Chunk Size Study**

In [ ]:
# A longer document to chunk (>500 words)
long_document = """
Neural networks are computational models inspired by the structure of the human brain.
They consist of layers of interconnected nodes, or neurons, that process information using
connectionist approaches. The basic unit is the perceptron, which takes weighted inputs,
applies an activation function, and produces an output. Deep neural networks stack many
such layers, allowing the model to learn hierarchical representations of data.

Training a neural network involves a forward pass, where input data flows through the
network to produce a prediction, and a backward pass, where gradients of the loss function
are propagated back through the network using backpropagation. The weights are updated
using an optimization algorithm such as stochastic gradient descent or Adam. The learning
rate controls how large each update step is, and choosing it carefully is critical for
convergence.

Convolutional neural networks, or CNNs, are specialized for grid-like data such as images.
They use convolutional filters that slide over the input, detecting local patterns like
edges and textures in early layers and complex shapes in deeper layers. Pooling layers
reduce spatial dimensions, making the representation more compact. CNNs form the backbone
of most modern computer vision systems.

Recurrent neural networks, or RNNs, are designed for sequential data. They maintain a
hidden state that is updated at each time step, allowing the network to carry information
forward across a sequence. However, vanilla RNNs suffer from the vanishing gradient problem,
which makes learning long-range dependencies difficult. Long Short-Term Memory networks,
or LSTMs, address this with gating mechanisms that control what information is stored,
forgotten, and output at each step.

Transformer architectures have largely replaced RNNs for sequence modeling tasks. Instead
of processing tokens sequentially, transformers use self-attention to relate every token
to every other token in the sequence in parallel. This makes them highly efficient on
modern hardware. The positional encoding adds information about token order, since
self-attention itself is permutation-invariant. Transformers are the foundation of
large language models like BERT and GPT.

Regularization techniques are used to prevent overfitting, where a model performs well on
training data but poorly on unseen data. Common methods include dropout, which randomly
deactivates neurons during training, L2 regularization which penalizes large weights, and
early stopping which halts training when validation performance stops improving. Data
augmentation artificially expands the training set by applying transformations such as
rotation, flipping, and cropping to existing samples.
"""

def chunk_document(text: str, chunk_size_words: int) -> list[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size_words):
        chunk = " ".join(words[i:i + chunk_size_words])
        chunks.append(chunk)
    return chunks

test_query = "how do transformers handle sequential data?"

print("Bonus 2 — Chunk Size Study\n")
print(f"Query: '{test_query}'\n")

for chunk_size in [50, 100, 200]:
    chunks = chunk_document(long_document, chunk_size)

    # Build a mini BM25 index on just these chunks
    tokenized_chunks = [c.lower().split() for c in chunks]
    bm25_chunks = BM25Okapi(tokenized_chunks)

    # Also SBERT encode chunks
    chunk_embs = retriever.sbert.encode(chunks, convert_to_numpy=True)
    query_emb = retriever.sbert.encode([test_query], convert_to_numpy=True)
    from sklearn.metrics.pairwise import cosine_similarity
    sbert_scores = cosine_similarity(chunk_embs, query_emb).flatten()
    top_sbert_id = int(np.argsort(sbert_scores)[::-1][0])

    bm25_scores = bm25_chunks.get_scores(test_query.lower().split())
    top_bm25_id = int(np.argsort(bm25_scores)[::-1][0])

    print(f"--- Chunk size: {chunk_size} words → {len(chunks)} chunks ---")
    print(f"  SBERT top chunk: '{chunks[top_sbert_id][:120]}...'")
    print(f"  BM25  top chunk: '{chunks[top_bm25_id][:120]}...'\n")
    print(f"  SBERT score: {sbert_scores[top_sbert_id]:.4f}")
    print(f"  BM25 score: {bm25_scores[top_bm25_id]:.4f}")

Bonus 2 — Chunk Size Study

Query: 'how do transformers handle sequential data?'

--- Chunk size: 50 words → 8 chunks ---
  SBERT top chunk: 'what information is stored, forgotten, and output at each step. Transformer architectures have largely replaced RNNs for...'
  BM25  top chunk: 'weights are updated using an optimization algorithm such as stochastic gradient descent or Adam. The learning rate contr...'

  SBERT score: 0.6701
  BM25 score: 1.5911
--- Chunk size: 100 words → 4 chunks ---
  SBERT top chunk: 'They maintain a hidden state that is updated at each time step, allowing the network to carry information forward across...'
  BM25  top chunk: 'weights are updated using an optimization algorithm such as stochastic gradient descent or Adam. The learning rate contr...'

  SBERT score: 0.5129
  BM25 score: 1.6753
--- Chunk size: 200 words → 2 chunks ---
  SBERT top chunk: 'They maintain a hidden state that is updated at each time step, allowing the network to carry information fo

## Bonus 2 — Chunk Size Study Observations

**Query:** "how do transformers handle sequential data?"

### SBERT Results

- **50 words:** SBERT retrieved a chunk that begins mid-sentence ("what information is stored, forgotten...") but still captures the transition from LSTMs to Transformers. The SBERT score was highest here (0.6701), meaning the smaller, focused chunk had a stronger semantic signal.
- **100 words:** SBERT retrieved the RNN/LSTM chunk instead — relevant to sequential data but not the best answer, since it focuses on RNNs rather than transformers. Score dropped to 0.5129.
- **200 words:** Same RNN/LSTM chunk retrieved, score dropped further to 0.4918. The transformer-specific content is now diluted inside a larger chunk that also covers RNNs, LSTMs, and other topics, weakening the semantic match.

**Takeaway for SBERT:** Smaller chunks preserve a stronger, more focused semantic signal. As chunk size grows, relevant content gets diluted by surrounding unrelated text, and SBERT scores consistently decrease.

### BM25 Results

- **50 words:** BM25 retrieved the wrong chunk entirely — an optimization/gradient descent paragraph with no relevance to the query. BM25 likely matched on incidental word overlap.
- **100 words:** Same wrong chunk retrieved, but with a slightly higher score (1.6753 vs 1.5911). More words per chunk means more chances for incidental keyword matches, which can hurt BM25 precision.
- **200 words:** BM25 correctly retrieved the RNN/LSTM/Transformer chunk, and both retrievers agreed for the first time. However, the BM25 score is 0.0000 — this happens because at 200 words the chunk is so large that term frequencies are normalized down to near zero by BM25's length penalty.

**Takeaway for BM25:** BM25 is unreliable at small chunk sizes (too few words to match on) and gets penalized at very large chunk sizes (length normalization). It performed best at 100 words for keyword matching, but still retrieved the wrong document.

### Overall Conclusion

| Chunk Size | SBERT Result | BM25 Result | Agreement? |
|---|---|---|---|
| 50 words | ✅ Transformer-relevant chunk (highest score) | ❌ Wrong chunk | ❌ No |
| 100 words | ⚠️ RNN chunk (related but not best) | ❌ Wrong chunk | ❌ No |
| 200 words | ⚠️ RNN chunk (diluted) | ⚠️ RNN chunk (score=0) | ✅ Yes |

50-word chunks gave SBERT the best retrieval quality for this query, confirming that smaller chunks preserve semantic focus. However, chunks that are too small risk splitting a coherent idea across boundaries (as seen with the mid-sentence chunk at 50 words). A chunk size of around 100 words is a practical balance — enough context for both retrievers to work with, without over-diluting the semantic signal.

## **Bonus Challenge 3 — ColBERT as Third Retriever in Hybrid RRF**

In [ ]:
def colbert_maxsim_scores(query: str, corpus_docs: list[str]) -> np.ndarray:
    """
    ColBERT-style MaxSim scoring using SBERT token embeddings.
    For each query token embedding, find the max similarity to any doc token embedding.
    Final score = mean of per-token max similarities.
    """
    # Encode query and docs at the token level
    # We use SBERT's underlying tokenizer + mean pooling per token via encode()
    # Since sentence-transformers returns sentence embeddings, we simulate token-level by splitting into word-level sub-phrases and encoding each
    query_tokens = query.lower().split()
    query_token_embs = retriever.sbert.encode(query_tokens, convert_to_numpy=True)  # (q_len, dim)

    scores = []
    for doc in corpus_docs:
        doc_tokens = doc.lower().split()
        if len(doc_tokens) == 0:
            scores.append(0.0)
            continue
        doc_token_embs = retriever.sbert.encode(doc_tokens, convert_to_numpy=True)  # (d_len, dim)

        # MaxSim: for each query token, find max cosine sim to any doc token
        # sim matrix: (q_len, d_len)
        from sklearn.metrics.pairwise import cosine_similarity
        sim_matrix = cosine_similarity(query_token_embs, doc_token_embs)
        max_sims = sim_matrix.max(axis=1)  # max over doc tokens for each query token
        scores.append(float(max_sims.mean()))

    return np.array(scores)


def three_way_hybrid_retrieve(query: str, top_k: int = 5, k: int = 60) -> list[dict]:
    """
    Hybrid retrieval fusing BM25 + SBERT + ColBERT with 3-way RRF.
    """
    # BM25
    bm25_scores = retriever.bm25.get_scores(query.lower().split())
    bm25_ranked = np.argsort(bm25_scores)[::-1]
    bm25_rank = {doc_id: rank for rank, doc_id in enumerate(bm25_ranked)}

    # SBERT
    query_emb = retriever.sbert.encode([query], convert_to_numpy=True)
    from sklearn.metrics.pairwise import cosine_similarity
    sbert_scores = cosine_similarity(retriever.doc_embeddings, query_emb).flatten()
    sbert_ranked = np.argsort(sbert_scores)[::-1]
    sbert_rank = {doc_id: rank for rank, doc_id in enumerate(sbert_ranked)}

    # ColBERT MaxSim
    print("  Computing ColBERT MaxSim scores (slower on CPU)...")
    colbert_scores = colbert_maxsim_scores(query, corpus)
    colbert_ranked = np.argsort(colbert_scores)[::-1]
    colbert_rank = {doc_id: rank for rank, doc_id in enumerate(colbert_ranked)}

    # 3-way RRF (equal weights)
    rrf_scores = {}
    for doc_id in range(len(corpus)):
        r_bm25 = bm25_rank.get(doc_id, len(corpus))
        r_sbert = sbert_rank.get(doc_id, len(corpus))
        r_colbert = colbert_rank.get(doc_id, len(corpus))
        rrf_scores[doc_id] = (
            1 / (k + r_bm25) +
            1 / (k + r_sbert) +
            1 / (k + r_colbert)
        )

    top_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:top_k]
    return [
        {
            "doc_id": i,
            "rrf_score": rrf_scores[i],
            "bm25_rank": bm25_rank[i],
            "sbert_rank": sbert_rank[i],
            "colbert_rank": colbert_rank[i],
            "text": corpus[i],
        }
        for i in top_ids
    ]


# Test it
print("Bonus 3 — ColBERT as Third Retriever\n")
test_q = "how do transformers encode meaning?"
print(f"Query: '{test_q}'\n")

print("[2-way Hybrid: BM25 + SBERT]")
two_way = retriever.retrieve(test_q, top_k=3)
for r in two_way:
    print(f"  [{r['doc_id']}] bm25_rank={r['bm25_rank']} sbert_rank={r['sbert_rank']} | {r['text'][:80]}...")

print("\n[3-way Hybrid: BM25 + SBERT + ColBERT]")
three_way = three_way_hybrid_retrieve(test_q, top_k=3)
for r in three_way:
    print(f"  [{r['doc_id']}] bm25={r['bm25_rank']} sbert={r['sbert_rank']} colbert={r['colbert_rank']} | {r['text'][:80]}...")

Bonus 3 — ColBERT as Third Retriever

Query: 'how do transformers encode meaning?'

[2-way Hybrid: BM25 + SBERT]
  [10] bm25_rank=1 sbert_rank=1 | BERT uses a masked language modeling objective to pretrain bidirectional represe...
  [9] bm25_rank=2 sbert_rank=5 | Word2Vec learns dense vector representations of words by predicting surrounding ...
  [11] bm25_rank=0 sbert_rank=8 | The Scaled Dot-Product Attention formula is: Attention(Q, K, V) = softmax(QK^T /...

[3-way Hybrid: BM25 + SBERT + ColBERT]
  Computing ColBERT MaxSim scores (slower on CPU)...
  [10] bm25=1 sbert=1 colbert=3 | BERT uses a masked language modeling objective to pretrain bidirectional represe...
  [9] bm25=2 sbert=5 colbert=1 | Word2Vec learns dense vector representations of words by predicting surrounding ...
  [0] bm25=11 sbert=0 colbert=0 | The transformer architecture uses self-attention to weigh the importance of each...


## Bonus 3 — ColBERT as Third Retriever Observations

**Query:** "how do transformers encode meaning?"

### 2-way Hybrid (BM25 + SBERT)

The top 3 results were `doc_id=10` (BERT), `doc_id=9` (Word2Vec), and `doc_id=11` (Scaled Dot-Product formula) — none of which directly answer the query. BERT and Word2Vec are *related* to representation learning but don't explain how transformers encode meaning. The formula document (`doc_id=11`) ranked first by BM25 (rank 0) due to keyword overlap but is irrelevant to the semantic intent of the query. This is a clear retrieval failure — the most relevant document (`doc_id=0`, the self-attention doc) was missed entirely.

### 3-way Hybrid (BM25 + SBERT + ColBERT)

Adding ColBERT changed the results significantly. `doc_id=0` (transformer self-attention) entered the top 3, ranked **ColBERT rank 0** and **SBERT rank 0** — both retrievers agree it's the best match. Despite BM25 ranking it last (rank 11), the RRF fusion pulled it into the top 3 because ColBERT and SBERT strongly agreed on its relevance.

`doc_id=9` (Word2Vec) moved up to second place, driven by **ColBERT rank 1** — ColBERT's token-level MaxSim likely matched query tokens like "encode" and "meaning" to similar words in the Word2Vec document ("vector representations", "context words"). This is a borderline result — Word2Vec is loosely relevant but not a direct answer.

### Key Takeaway

| Doc | 2-way rank | 3-way rank | Actually relevant? |
|---|---|---|---|
| BERT (`doc_id=10`) | #1 | #1 | ⚠️ Loosely |
| Word2Vec (`doc_id=9`) | #2 | #2 | ⚠️ Loosely |
| Formula (`doc_id=11`) | #3 | ❌ Dropped out | ❌ No |
| Self-attention (`doc_id=0`) | ❌ Missing | #3 | ✅ Yes |

The most important improvement is that `doc_id=0` — the most directly relevant document — was completely absent from the 2-way results but appeared in the 3-way top 3 thanks to ColBERT. ColBERT's token-level MaxSim scoring captured fine-grained word-level matches that sentence-level SBERT missed, and overrode BM25's poor ranking of that document. This demonstrates that ColBERT adds genuine complementary signal beyond what BM25 and SBERT provide, especially for queries where the relevant document shares key individual words with the query but not overall sentence-level similarity.